# Verify Module 3: LCFE (Landmark Coordinates Feature Extraction)

Runs Module 1's nose-tip-aligned coordinate output through
`fusion_avsr.models.landmark.lcfe.LCFE` for a clip in each of the three
sources: `(38, 2, T)` aligned coordinates -> `(38, 256, T)` features,
matching LMFE's output shape so the two can later be concatenated.

**What "looks right" means:**
- Output shape is exactly `(38, 256, T)` for every source, `T` unchanged.
- No `NaN`/`Inf` values.
- A backward pass reaches every LCFE parameter.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent  # adjust if running from somewhere else
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# TODO: fill in correct paths (same convention as notebook 01)
PROJECT_NAME = "your_project_name"
DATASET_ROOT = Path(f"/scratch/{PROJECT_NAME}/datasets")
LRS3_ROOT = DATASET_ROOT / "lrs3"

grid_all_path = "kaggle_lipnet/datasets/jedidiahangekouakou/grid-corpus-dataset-for-training-lipnet/versions/1/data"
GRID_ROOT = DATASET_ROOT / grid_all_path
GRID_LANDMARKS_ROOT = DATASET_ROOT / "grid_landmarks"

LRS3_TRAINVAL_LANDMARKS_ROOT = LRS3_ROOT / "landmarks" / "LRS3_landmarks" / "trainval"
LRS3_TRAINVAL_VIDEO_ROOT = LRS3_ROOT / "ainncy" / "trainval"

LRS3_TEST_VIDEO_ROOT = LRS3_ROOT / "test"
LRS3_TEST_LANDMARKS_ROOT = LRS3_ROOT / "landmarks" / "LRS3_landmarks" / "test"

AUDIO_OUTPUT_DIR = DATASET_ROOT / "extracted_audio"
LIMIT = 5  # a handful of clips per source, not the whole dataset

SEED = 42

In [ ]:
import pickle

import numpy as np
import pandas as pd
import torch
from torchcodec.decoders import VideoDecoder

from fusion_avsr.data.manifest_builder import build_grid_manifest, build_lrs3_manifest, load_or_build_manifest
from fusion_avsr.data.paths import MANIFEST_DIR

# These are cached under MANIFEST_DIR: built once (here or by notebook 01,
# whichever runs first), loaded from the cached CSV every time after.
grid_manifest = load_or_build_manifest(
    MANIFEST_DIR / "grid_manifest.csv", build_grid_manifest,
    grid_root=GRID_ROOT, landmarks_root=GRID_LANDMARKS_ROOT,
    audio_output_dir=AUDIO_OUTPUT_DIR, limit=LIMIT,
)
lrs3_trainval_manifest = load_or_build_manifest(
    MANIFEST_DIR / "lrs3_trainval_manifest.csv", build_lrs3_manifest,
    video_root=LRS3_TRAINVAL_VIDEO_ROOT, audio_output_dir=AUDIO_OUTPUT_DIR,
    landmarks_root=LRS3_TRAINVAL_LANDMARKS_ROOT, source="lrs3_trainval", limit=LIMIT,
)
lrs3_test_manifest = load_or_build_manifest(
    MANIFEST_DIR / "lrs3_test_manifest.csv", build_lrs3_manifest,
    video_root=LRS3_TEST_VIDEO_ROOT, audio_output_dir=AUDIO_OUTPUT_DIR,
    landmarks_root=LRS3_TEST_LANDMARKS_ROOT, source="lrs3_test", limit=LIMIT,
)

MANIFESTS = {
    "grid": grid_manifest,
    "lrs3_trainval": lrs3_trainval_manifest,
    "lrs3_test": lrs3_test_manifest,
}
for name, m in MANIFESTS.items():
    print(name, m.shape)

In [ ]:
def load_one_clip(manifest, seed=SEED):
    """Pick one random row and decode its frames + landmarks."""
    row = manifest.sample(n=1, random_state=seed).iloc[0]
    decoder = VideoDecoder(row["video_path"], dimension_order="NHWC")
    frames = np.stack([decoder[t].numpy() for t in range(len(decoder))])
    with open(row["landmark_path"], "rb") as f:
        landmarks = pickle.load(f)
    n = min(len(frames), len(landmarks))
    return row["sample_id"], frames[:n], landmarks[:n]

In [ ]:
from fusion_avsr.models.landmark.lrlp import align_to_nose_tip, extract_lrlp_sequence

def get_module1_outputs(manifest, seed=SEED):
    sample_id, frames, landmarks = load_one_clip(manifest, seed=seed)
    patches, raw_coords, valid_mask = extract_lrlp_sequence(frames, landmarks)
    aligned_coords = align_to_nose_tip(raw_coords, landmarks)
    return sample_id, patches, raw_coords, aligned_coords, valid_mask

In [ ]:
import torch

from fusion_avsr.models.landmark.lcfe import LCFE, LCFE_OUTPUT_CHANNELS

lcfe = LCFE()
print(lcfe)

## Run LCFE on one real clip from each source

In [ ]:
for source_name, manifest in MANIFESTS.items():
    sample_id, patches, raw_coords, aligned_coords, valid_mask = get_module1_outputs(manifest)
    # LCFE expects (B, K, 2, T); align_to_nose_tip returns (K, T, 2).
    coords_tensor = torch.from_numpy(aligned_coords).float().permute(0, 2, 1).unsqueeze(0)

    output = lcfe(coords_tensor)
    print(f"{source_name} ({sample_id}): input={tuple(coords_tensor.shape)} -> output={tuple(output.shape)}")

    assert output.shape[-1] == coords_tensor.shape[-1], "T must be unchanged"
    assert output.shape[2] == LCFE_OUTPUT_CHANNELS
    assert torch.isfinite(output).all(), "found NaN/Inf in LCFE output"

## Backward pass reaches every parameter

In [ ]:
sample_id, patches, raw_coords, aligned_coords, valid_mask = get_module1_outputs(MANIFESTS["grid"])
coords_tensor = torch.from_numpy(aligned_coords).float().permute(0, 2, 1).unsqueeze(0)

output = lcfe(coords_tensor)
output.sum().backward()

missing_grad = [name for name, p in lcfe.named_parameters() if p.grad is None]
print("parameters with no gradient:", missing_grad)
assert not missing_grad

## Checklist

- [ ] `output.shape == (1, 38, 256, T)` for GRID, LRS3-trainval, AND LRS3-test, with `T` matching the input.
- [ ] No assertion errors.
- [ ] No parameters missing a gradient after the backward pass.